Calculates the relative difference between NHDA and RA for development characteristics and difference for residential building type share.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path


# ============================================================================
# CONFIG
# ============================================================================

INPUT_FILE  = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure.gpkg"
OUTPUT_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences.gpkg"
OUTPUT_LAYER = "Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences"
PLOT_DIR    = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Plots\BuildingStructure_Distribution"

NUMERIC_METRICS = [
    'built_up_ratio',
    'building_volume_density',
    'building_count',
    'building_density',
    'avg_building_height',
    'avg_building_footprint',
]

METRIC_LABELS = {
    'built_up_ratio':          'Built-up Ratio [%]',
    'building_volume_density': 'Building Volume Density [m³/m²]',
    'building_count':          'Building Count',
    'building_density':        'Building Density [1/ha]',
    'avg_building_height':     'Avg. Building Height [m]',
    'avg_building_footprint':  'Avg. Building Footprint [m²]',
}

# Residential building subclass shares -> absolute difference (percentage points),
# NOT relative difference, since these are already shares in %.
RES_SUBCLASS_METRICS = [
    'res_subclass_share_mfh_ab',
    'res_subclass_share_sbd',
    'res_subclass_share_sfh_db',
    'res_subclass_share_tb',
]

RES_SUBCLASS_LABELS = {
    'res_subclass_share_mfh_ab': 'MFH-AB Share [%]',
    'res_subclass_share_sbd':    'SBD Share [%]',
    'res_subclass_share_sfh_db': 'SFH-DB Share [%]',
    'res_subclass_share_tb':     'TB Share [%]',
}


# ============================================================================
# LOAD
# ============================================================================

print("Loading input file...")
gdf = gpd.read_file(INPUT_FILE)
print(f"  → {len(gdf)} features")

nhda_gdf = gdf[gdf['type'] == 'NHDA'].copy()
ra_gdf   = gdf[gdf['type'] == 'RA'].copy()
print(f"  → {len(nhda_gdf)} NHDA rows, {len(ra_gdf)} RA rows")

Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)


# ============================================================================
# PLOT — NUMERIC_METRICS (relative difference -> reldiff_<metric>)
# ============================================================================

for metric in NUMERIC_METRICS:
    nhda_col = f'nhda_{metric}'
    ra_col   = f'ra_{metric}'

    if nhda_col not in gdf.columns or ra_col not in gdf.columns:
        print(f"  ⚠️  Skipping {metric} – columns not found")
        continue

    # Use NHDA rows as the "per-pair" unit (each nhda_id appears once)
    nhda_vals = pd.to_numeric(nhda_gdf[nhda_col], errors='coerce').dropna()
    ra_vals   = pd.to_numeric(nhda_gdf[ra_col],   errors='coerce').dropna()

    # Relative difference per pair (only where both exist)
    both_valid = nhda_gdf[[nhda_col, ra_col]].apply(pd.to_numeric, errors='coerce').dropna()
    nhda_v = both_valid[nhda_col]
    ra_v   = both_valid[ra_col]
    mean_v = (nhda_v + ra_v) / 2.0
    rel_diff = np.where(mean_v == 0, np.nan, (nhda_v - ra_v) / mean_v * 100.0)
    rel_diff = pd.Series(rel_diff, index=both_valid.index)

    # Store as new column on nhda_gdf (one value per pair)
    reldiff_col = f'reldiff_{metric}'
    nhda_gdf.loc[both_valid.index, reldiff_col] = rel_diff

    rel_diff = rel_diff.dropna()

    label = METRIC_LABELS.get(metric, metric)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{label}\nNHDA vs. RA Distribution", fontsize=13, fontweight='bold')

    # --- Left: overlapping histograms ---
    ax = axes[0]
    bins = np.histogram_bin_edges(
        pd.concat([nhda_vals, ra_vals]).dropna(), bins=40
    )
    ax.hist(nhda_vals, bins=bins, alpha=0.6, color='steelblue',  label='NHDA', edgecolor='white', linewidth=0.4)
    ax.hist(ra_vals,   bins=bins, alpha=0.6, color='darkorange', label='RA',   edgecolor='white', linewidth=0.4)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title('Value Distribution', fontsize=11)
    ax.legend()

    # Descriptive stats box
    stats_text = (
        f"NHDA  mean={nhda_vals.mean():.2f}  median={nhda_vals.median():.2f}\n"
        f"RA    mean={ra_vals.mean():.2f}  median={ra_vals.median():.2f}"
    )
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    # --- Right: relative difference histogram ---
    ax2 = axes[1]
    p5, p95 = np.percentile(rel_diff, [5, 95])
    bins_rd = np.linspace(max(rel_diff.min(), -200), min(rel_diff.max(), 200), 60)
    ax2.hist(rel_diff, bins=bins_rd, color='mediumpurple', edgecolor='white', linewidth=0.4)
    ax2.axvline(0,    color='black',     linewidth=1.5, linestyle='--', label='0%')
    ax2.axvline( 10,  color='green',     linewidth=1.2, linestyle=':',  label='±10%')
    ax2.axvline(-10,  color='green',     linewidth=1.2, linestyle=':')
    ax2.axvline( 20,  color='goldenrod', linewidth=1.2, linestyle=':',  label='±20%')
    ax2.axvline(-20,  color='goldenrod', linewidth=1.2, linestyle=':')
    ax2.set_xlabel('Relative Difference (NHDA − RA) / mean × 100 [%]', fontsize=10)
    ax2.set_ylabel('Count', fontsize=10)
    ax2.set_title('Relative Difference per Pair', fontsize=11)
    ax2.legend(fontsize=8)

    n_total  = len(rel_diff)
    n_10     = (rel_diff.abs() <= 10).sum()
    n_20     = (rel_diff.abs() <= 20).sum()
    stats2   = (
        f"n pairs = {n_total}\n"
        f"within ±10%: {n_10} ({n_10/n_total*100:.1f}%)\n"
        f"within ±20%: {n_20} ({n_20/n_total*100:.1f}%)\n"
        f"median diff: {rel_diff.median():.1f}%\n"
        f"std diff:    {rel_diff.std():.1f}%"
    )
    ax2.text(0.98, 0.97, stats2, transform=ax2.transAxes,
             fontsize=8, va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    plt.tight_layout()

    # Save
    out_path = Path(PLOT_DIR) / f"dist_{metric}.png"
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ {metric}: plot saved → {out_path.name}")


# ============================================================================
# PLOT — RES_SUBCLASS_METRICS (absolute difference -> diff_<subclass>)
# ============================================================================

for metric in RES_SUBCLASS_METRICS:
    nhda_col = f'nhda_{metric}'
    ra_col   = f'ra_{metric}'

    if nhda_col not in gdf.columns or ra_col not in gdf.columns:
        print(f"  ⚠️  Skipping {metric} – columns not found")
        continue

    nhda_vals = pd.to_numeric(nhda_gdf[nhda_col], errors='coerce').dropna()
    ra_vals   = pd.to_numeric(nhda_gdf[ra_col],   errors='coerce').dropna()

    # Absolute difference per pair (percentage points), only where both exist
    both_valid = nhda_gdf[[nhda_col, ra_col]].apply(pd.to_numeric, errors='coerce').dropna()
    nhda_v = both_valid[nhda_col]
    ra_v   = both_valid[ra_col]
    abs_diff = nhda_v - ra_v

    # Store as new column on nhda_gdf (one value per pair)
    diff_col = f'diff_{metric}'
    nhda_gdf.loc[both_valid.index, diff_col] = abs_diff

    label = RES_SUBCLASS_LABELS.get(metric, metric)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{label}\nNHDA vs. RA Distribution", fontsize=13, fontweight='bold')

    # --- Left: overlapping histograms ---
    ax = axes[0]
    bins = np.histogram_bin_edges(
        pd.concat([nhda_vals, ra_vals]).dropna(), bins=40
    )
    ax.hist(nhda_vals, bins=bins, alpha=0.6, color='steelblue',  label='NHDA', edgecolor='white', linewidth=0.4)
    ax.hist(ra_vals,   bins=bins, alpha=0.6, color='darkorange', label='RA',   edgecolor='white', linewidth=0.4)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title('Value Distribution', fontsize=11)
    ax.legend()

    stats_text = (
        f"NHDA  mean={nhda_vals.mean():.2f}  median={nhda_vals.median():.2f}\n"
        f"RA    mean={ra_vals.mean():.2f}  median={ra_vals.median():.2f}"
    )
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    # --- Right: absolute difference histogram ---
    ax2 = axes[1]
    bins_ad = np.linspace(abs_diff.min(), abs_diff.max(), 60)
    ax2.hist(abs_diff, bins=bins_ad, color='mediumseagreen', edgecolor='white', linewidth=0.4)
    ax2.axvline(0,   color='black', linewidth=1.5, linestyle='--', label='0 pp')
    ax2.axvline( 5,  color='green', linewidth=1.2, linestyle=':',  label='±5 pp')
    ax2.axvline(-5,  color='green', linewidth=1.2, linestyle=':')
    ax2.axvline( 10, color='goldenrod', linewidth=1.2, linestyle=':', label='±10 pp')
    ax2.axvline(-10, color='goldenrod', linewidth=1.2, linestyle=':')
    ax2.set_xlabel('Absolute Difference (NHDA − RA) [pp]', fontsize=10)
    ax2.set_ylabel('Count', fontsize=10)
    ax2.set_title('Absolute Difference per Pair', fontsize=11)
    ax2.legend(fontsize=8)

    n_total = len(abs_diff)
    n_5     = (abs_diff.abs() <= 5).sum()
    n_10    = (abs_diff.abs() <= 10).sum()
    stats2  = (
        f"n pairs = {n_total}\n"
        f"within ±5pp: {n_5} ({n_5/n_total*100:.1f}%)\n"
        f"within ±10pp: {n_10} ({n_10/n_total*100:.1f}%)\n"
        f"median diff: {abs_diff.median():.1f} pp\n"
        f"std diff:    {abs_diff.std():.1f} pp"
    )
    ax2.text(0.98, 0.97, stats2, transform=ax2.transAxes,
             fontsize=8, va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    plt.tight_layout()

    out_path = Path(PLOT_DIR) / f"dist_{metric}.png"
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ {metric}: plot saved → {out_path.name}")


print(f"\nAll plots saved to: {PLOT_DIR}")
print(f"New columns added to nhda_gdf: "
      f"{[c for c in nhda_gdf.columns if c.startswith('reldiff_') or c.startswith('diff_')]}")


# ============================================================================
# ADD DIFFERENCE COLUMNS TO FULL GEODATAFRAME
# ============================================================================

# Collect all newly created difference columns
difference_cols = [
    col for col in nhda_gdf.columns
    if col.startswith("reldiff_") or col.startswith("diff_")
]

print("\nAdding difference columns to full GeoDataFrame...")
print(f"  → Columns: {difference_cols}")

# Keep one row per NHDA–RA pair
pair_differences = nhda_gdf[
    ["nhda_id"] + difference_cols
].copy()

# Remove duplicate NHDA IDs (should not occur, but ensures uniqueness)
pair_differences = pair_differences.drop_duplicates(
    subset="nhda_id"
)

# Remove existing difference columns if they already exist
existing_difference_cols = [
    col for col in difference_cols
    if col in gdf.columns
]

if existing_difference_cols:
    gdf = gdf.drop(columns=existing_difference_cols)

# Join the pairwise difference values back to all NHDA and RA rows
gdf = gdf.merge(
    pair_differences,
    on="nhda_id",
    how="left",
    validate="many_to_one"
)

# Convert back to a GeoDataFrame
gdf = gpd.GeoDataFrame(
    gdf,
    geometry="geometry",
    crs=nhda_gdf.crs
)

# ============================================================================
# SAVE OUTPUT
# ============================================================================

output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

gdf.to_file(
    OUTPUT_FILE,
    layer=OUTPUT_LAYER,
    driver="GPKG"
)

print(f"\n✓ GeoPackage saved: {OUTPUT_FILE}")
print(f"✓ Layer: {OUTPUT_LAYER}")
print(f"✓ Added {len(difference_cols)} difference columns")